# Hero specialty feature vectors

This notebook turns `analysis/hero_features.json` into reusable numeric hero representations. It implements two encodings:

- **Compact (12-D):** lane one-hot + damage multi-hot + normalized ordinal specialty values.
- **Thermometer (15-D, recommended default):** preserves the ordering of control, mobility, and healing without assuming their steps are equally spaced.

It also provides lookup and draft-context helpers that can be joined to historical KPL draft data later.

## 1. Setup and load the source data

The notebook uses only the Python standard library, so it can run in the existing environment without adding a dependency.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json


def find_repo_root(start: Path) -> Path:
    """Find the repository root whether the notebook starts from it or analysis/."""
    for candidate in (start, *start.parents):
        if (candidate / 'analysis' / 'hero_features.json').is_file():
            return candidate
    raise FileNotFoundError('Could not find analysis/hero_features.json above the current directory.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
ANALYSIS_DIR = REPO_ROOT / 'analysis'
SOURCE_PATH = ANALYSIS_DIR / 'hero_features.json'
COVERAGE_PATH = ANALYSIS_DIR / 'hero_feature_coverage.json'
EXPORT_PATH = ANALYSIS_DIR / 'hero_specialty_vectors_thermometer.json'

heroes = json.loads(SOURCE_PATH.read_text(encoding='utf-8'))
coverage = json.loads(COVERAGE_PATH.read_text(encoding='utf-8'))

print(f'Loaded {len(heroes)} specialty profiles from {SOURCE_PATH.relative_to(REPO_ROOT)}')

## 2. Inspect coverage before modeling

`hero_name` and `hero_id` are lookup keys, not model inputs. Resolve missing IDs before joining to match data; do not turn a missing ID into zero.

In [ ]:
unresolved_ids = [hero['hero_name'] for hero in heroes if hero['hero_id'] is None]
lane_counts = Counter(hero['primary_lane'] for hero in heroes)
damage_counts = Counter(damage for hero in heroes for damage in hero['damage_types'])

print('Profiles:', len(heroes))
print('Matched hero IDs:', len(heroes) - len(unresolved_ids))
print('Unresolved source IDs:', unresolved_ids)
print('Catalog heroes missing profiles:', coverage['catalog_heroes_missing_features'])
print('Lane counts:', dict(sorted(lane_counts.items())))
print('Damage-type counts:', dict(sorted(damage_counts.items())))

## 3. Define the vector schemas

Damage is a multi-label attribute, so it is encoded as three independent bits. The compact scheme treats the three `0–2` specialties as normalized ordinals. The thermometer scheme uses two threshold bits per specialty: for example, strong control (`2`) becomes `[1, 1]`, weak control (`1`) becomes `[1, 0]`, and no control becomes `[0, 0]`.

In [ ]:
LANES = ('clash', 'mid', 'jungle', 'farm', 'roam')
DAMAGE_TYPES = ('physical', 'magic', 'true')

COMPACT_FEATURE_NAMES = (
    *(f'lane__{lane}' for lane in LANES),
    *(f'damage__{damage}' for damage in DAMAGE_TYPES),
    'control__normalized',
    'mobility__normalized',
    'has_unstoppable',
    'heal__normalized',
)

THERMOMETER_FEATURE_NAMES = (
    *(f'lane__{lane}' for lane in LANES),
    *(f'damage__{damage}' for damage in DAMAGE_TYPES),
    'control__at_least_weak',
    'control__strong',
    'mobility__at_least_small',
    'mobility__large',
    'has_unstoppable',
    'heal__at_least_self_or_conditional',
    'heal__reliable_ally_or_team',
)

assert len(COMPACT_FEATURE_NAMES) == 12
assert len(THERMOMETER_FEATURE_NAMES) == 15


def validate_hero(hero: dict) -> None:
    """Fail early if the controlled source vocabulary changes."""
    if hero['primary_lane'] not in LANES:
        raise ValueError(f"Unknown lane for {hero['hero_name']}: {hero['primary_lane']}")
    unknown_damage = set(hero['damage_types']) - set(DAMAGE_TYPES)
    if unknown_damage:
        raise ValueError(f"Unknown damage type for {hero['hero_name']}: {unknown_damage}")
    for specialty in ('control', 'mobility', 'heal'):
        if hero[specialty] not in (0, 1, 2):
            raise ValueError(f"{specialty} must be 0, 1, or 2 for {hero['hero_name']}")


def one_hot(value: str, vocabulary: tuple[str, ...]) -> list[float]:
    return [float(value == category) for category in vocabulary]


def multi_hot(values: list[str], vocabulary: tuple[str, ...]) -> list[float]:
    values = set(values)
    return [float(category in values) for category in vocabulary]


def compact_vector(hero: dict) -> list[float]:
    """Return the 12-D normalized ordinal representation."""
    validate_hero(hero)
    return [
        *one_hot(hero['primary_lane'], LANES),
        *multi_hot(hero['damage_types'], DAMAGE_TYPES),
        hero['control'] / 2.0,
        hero['mobility'] / 2.0,
        float(hero['has_unstoppable']),
        hero['heal'] / 2.0,
    ]


def thermometer(value: int) -> list[float]:
    return [float(value >= 1), float(value >= 2)]


def thermometer_vector(hero: dict) -> list[float]:
    """Return the 15-D monotonic/threshold representation."""
    validate_hero(hero)
    return [
        *one_hot(hero['primary_lane'], LANES),
        *multi_hot(hero['damage_types'], DAMAGE_TYPES),
        *thermometer(hero['control']),
        *thermometer(hero['mobility']),
        float(hero['has_unstoppable']),
        *thermometer(hero['heal']),
    ]

## 4. Build and validate the per-hero feature table

The source fields stay alongside the vectors for traceability. The 15-D thermometer representation is set as the default; keep the compact representation when a small continuous vector is more useful.

In [ ]:
feature_rows = []
for hero in heroes:
    compact = compact_vector(hero)
    thermometer_features = thermometer_vector(hero)
    feature_rows.append({
        'hero_id': hero['hero_id'],
        'hero_name': hero['hero_name'],
        'feature_known': True,
        'compact_vector': compact,
        'thermometer_vector': thermometer_features,
    })

assert len(feature_rows) == len(heroes)
assert all(len(row['compact_vector']) == len(COMPACT_FEATURE_NAMES) for row in feature_rows)
assert all(len(row['thermometer_vector']) == len(THERMOMETER_FEATURE_NAMES) for row in feature_rows)
assert all(all(value in (0.0, 1.0) for value in row['thermometer_vector']) for row in feature_rows)

print(f'Built {len(feature_rows)} compact vectors with {len(COMPACT_FEATURE_NAMES)} features each.')
print(f'Built {len(feature_rows)} thermometer vectors with {len(THERMOMETER_FEATURE_NAMES)} features each.')

sample = next(row for row in feature_rows if row['hero_name'] == '吕布')
print('Sample: 吕布')
print(dict(zip(THERMOMETER_FEATURE_NAMES, sample['thermometer_vector'])))

## 5. Create safe lookup helpers

Use these helpers to join a draft record by canonical hero ID when available, or by exact hero name during cleanup. Unknown heroes return `None`; callers should add an explicit `feature_known` flag and a deliberate fallback instead of silently substituting an all-zero profile.

In [ ]:
VECTOR_SCHEMES = {
    'compact': ('compact_vector', COMPACT_FEATURE_NAMES),
    'thermometer': ('thermometer_vector', THERMOMETER_FEATURE_NAMES),
}
rows_by_name = {row['hero_name']: row for row in feature_rows}
rows_by_id = {row['hero_id']: row for row in feature_rows if row['hero_id'] is not None}


def get_feature_row(hero_ref: int | str) -> dict | None:
    if isinstance(hero_ref, int):
        return rows_by_id.get(hero_ref)
    if isinstance(hero_ref, str):
        return rows_by_name.get(hero_ref.strip())
    raise TypeError('hero_ref must be an integer hero ID or a hero-name string')


def get_hero_vector(hero_ref: int | str, scheme: str = 'thermometer') -> list[float] | None:
    if scheme not in VECTOR_SCHEMES:
        raise ValueError(f'Unknown scheme {scheme!r}; choose from {tuple(VECTOR_SCHEMES)}')
    row = get_feature_row(hero_ref)
    return None if row is None else row[VECTOR_SCHEMES[scheme][0]]


assert get_hero_vector('吕布') == sample['thermometer_vector']
assert get_hero_vector('not-a-hero') is None

print('吕布 feature vector:', get_hero_vector('吕布'))
print('Unknown feature vector:', get_hero_vector('not-a-hero'))

## 6. Turn a candidate pick and draft state into model features

A per-hero vector describes only the candidate. For a draft recommendation or win-probability model, concatenate the candidate representation with ally and opponent composition summaries. `sum` counts available traits, `max` marks whether a capability exists, and `ally_minus_enemy` expresses the current composition balance.

In [ ]:
def elementwise_sum(vectors: list[list[float]], width: int) -> list[float]:
    return [sum(vector[index] for vector in vectors) for index in range(width)]


def elementwise_max(vectors: list[list[float]], width: int) -> list[float]:
    return [max((vector[index] for vector in vectors), default=0.0) for index in range(width)]


def draft_context_features(
    candidate: int | str,
    allies: list[int | str],
    enemies: list[int | str],
    scheme: str = 'thermometer',
) -> tuple[tuple[str, ...], list[float]]:
    """Build a fixed-width candidate-in-context feature vector."""
    if scheme not in VECTOR_SCHEMES:
        raise ValueError(f'Unknown scheme {scheme!r}; choose from {tuple(VECTOR_SCHEMES)}')

    _, base_names = VECTOR_SCHEMES[scheme]
    candidate_vector = get_hero_vector(candidate, scheme)
    ally_vectors = [get_hero_vector(hero, scheme) for hero in allies]
    enemy_vectors = [get_hero_vector(hero, scheme) for hero in enemies]

    unknown = [
        hero
        for hero, vector in [(candidate, candidate_vector), *zip(allies, ally_vectors), *zip(enemies, enemy_vectors)]
        if vector is None
    ]
    if unknown:
        raise KeyError(f'No specialty vector for: {unknown}')

    width = len(base_names)
    ally_sum = elementwise_sum(ally_vectors, width)
    ally_max = elementwise_max(ally_vectors, width)
    enemy_sum = elementwise_sum(enemy_vectors, width)
    enemy_max = elementwise_max(enemy_vectors, width)
    balance = [ally_sum[index] - enemy_sum[index] for index in range(width)]

    names = tuple(
        [f'candidate__{name}' for name in base_names]
        + [f'ally_sum__{name}' for name in base_names]
        + [f'ally_max__{name}' for name in base_names]
        + [f'enemy_sum__{name}' for name in base_names]
        + [f'enemy_max__{name}' for name in base_names]
        + [f'ally_minus_enemy__{name}' for name in base_names]
    )
    values = candidate_vector + ally_sum + ally_max + enemy_sum + enemy_max + balance
    return names, values


context_names, context_values = draft_context_features(
    candidate='吕布',
    allies=['赵云', '小乔'],
    enemies=['鲁班七号', '张良'],
)

print(f'Draft-context width: {len(context_values)}')
print(dict(list(zip(context_names, context_values))[:15]))

## 7. Export a stable feature artifact (optional)

Set `WRITE_EXPORT` to `True` when a downstream training job should consume the thermometer vectors. The output records the schema and ordered feature names, so a model cannot silently misinterpret vector positions.

In [ ]:
WRITE_EXPORT = True

if WRITE_EXPORT:
    export = {
        'schema_version': 1,
        'vector_scheme': 'thermometer',
        'feature_names': list(THERMOMETER_FEATURE_NAMES),
        'rows': [
            {
                'hero_id': row['hero_id'],
                'hero_name': row['hero_name'],
                'feature_known': row['feature_known'],
                'vector': row['thermometer_vector'],
            }
            for row in feature_rows
        ],
    }
    EXPORT_PATH.write_text(json.dumps(export, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'Wrote {len(feature_rows)} vectors to {EXPORT_PATH.relative_to(REPO_ROOT)}')
else:
    print('Export disabled. Set WRITE_EXPORT = True to write the feature artifact.')

## 8. Modeling next steps

- Train a tabular model on the draft-context features above, using chronological splits so patch and meta shifts do not leak from future matches.
- Keep per-hero historical effects or learned hero embeddings alongside these vectors; specialties explain composition, while identity features capture kit-specific and meta effects.
- If you add multi-lane eligibility later, replace the single primary-lane one-hot with a multi-hot lane pool.
- Compare the compact and thermometer encodings by validation performance and calibration rather than choosing only on intuition.